# Spark Connect from an OpenShift AI workbench

Runs against the `SparkConnect` server from `deploy/openshift-ai/20-sparkconnect.yaml`.
Workbench image: any Python 3.12 Jupyter image. Run examples 1 and 2 first so there is data to explore.

The notebook needs **no** Spark installation, JVM or cluster credentials - only the thin client.

In [ ]:
# OpenShift AI workbenches install from Red Hat's Python index. In OpenShift AI 3.5 it offers
# pyspark-client up to 4.2.0, which is tested against this 4.3.0 server. With a PyPI mirror
# configured the same line installs 4.3.0. Restart the kernel after installing.
%pip install --quiet "pyspark-client>=4.2,<=4.3.0"

In [ ]:
import os
from pyspark.sql import SparkSession, functions as F

NAMESPACE = os.environ.get("SPARK_CONNECT_NAMESPACE", "spark-demo")   # namespace of the SparkConnect server
BUCKET = os.environ.get("S3_BUCKET", "spark-smoke")

spark = SparkSession.builder.remote(
    f"sc://spark-connect-service.{NAMESPACE}.svc.cluster.local:15002").getOrCreate()
print("Connected. Server Spark version:", spark.version)

If `getOrCreate()` **hangs** rather than failing, a NetworkPolicy is dropping the traffic: the workbench pod needs the label `opendatahub.io/workbenches=true` and its namespace `opendatahub.io/dashboard=true`.

In [ ]:
# Executors read the curated zone straight from S3; only the result comes back to the notebook.
txns = spark.read.parquet(f"s3a://{BUCKET}/examples/curated/transactions")
txns.printSchema()
txns.groupBy("category").agg(F.count("*").alias("txns"), F.round(F.avg("amount"), 2).alias("avg_amount")).orderBy("category").show()

In [ ]:
# A pandas UDF defined here, executed on the cluster's Python workers (Arrow).
import pandas as pd

@F.pandas_udf("string")
def band(amount: pd.Series) -> pd.Series:
    return pd.cut(amount, [0, 50, 200, 1e9], labels=["small", "medium", "large"]).astype(str)

txns.groupBy(band("amount").alias("band")).count().toPandas()

In [ ]:
# Spark SQL
txns.createOrReplaceTempView("txns")
spark.sql("""
  SELECT category, account_id, round(sum(amount), 2) AS spend
  FROM txns GROUP BY category, account_id ORDER BY spend DESC LIMIT 10
""").toPandas()

### Database access
JDBC credentials should come from a Secret mounted into the workbench as environment variables (data connection), never typed into a notebook.

In [ ]:
jdbc = dict(url=os.environ["JDBC_URL"], user=os.environ["JDBC_USER"], password=os.environ["JDBC_PASSWORD"])
summary = spark.read.format("jdbc").options(dbtable="daily_category_summary", **jdbc).load()
summary.orderBy("category").toPandas()

In [ ]:
spark.stop()